# Panadería Chapala · Solución Big Data con PySpark

Este cuaderno reproduce la solución Big Data del proyecto en Google Colab (o en Jupyter local):

1. Ingesta de los CSV exportados de SQL Server a la capa **bronce** (Parquet).
2. Transformación a **plata** (tipos y limpieza) y **oro** (modelo estrella particionado por año/mes).
3. Controles de calidad, consultas Spark SQL y experimento de tiempos (x1, x10, x50).

Ejecute las celdas en orden con *Entorno de ejecución → Ejecutar todas*.

## 1. Obtener el repositorio e instalar dependencias

In [ ]:
REPO_URL = "https://github.com/<usuario>/panaderia-chapala-bi-bigdata.git"   # <- reemplazar por la URL del grupo
import os
if not os.path.exists("panaderia-chapala-bi-bigdata"):
    !git clone --branch v1.0-final {REPO_URL} panaderia-chapala-bi-bigdata
%cd panaderia-chapala-bi-bigdata
!git log -1 --format="%H %ad %s"
!pip -q install -r requirements.txt

## 2. Pipeline: ingesta → transformación → calidad → consultas

In [ ]:
!python bigdata/run_pipeline.py --etapas ingesta,transformacion,calidad,consultas

## 3. Monitoreo: métricas por etapa y particiones generadas

In [ ]:
import json, pandas as pd, pathlib
metricas = pd.read_json("logs/metricas.jsonl", lines=True)
display(metricas.tail(40)[["ejecucion_id", "etapa", "tabla", "filas", "segundos", "estado"]])

print("Particiones de fact_ventas en la capa oro:")
for p in sorted(pathlib.Path("datalake/gold/fact_ventas").glob("anio=*/mes=*")):
    archivos = list(p.glob("*.parquet"))
    print(f"  {p.relative_to('datalake/gold')}: {len(archivos)} archivo(s), {sum(a.stat().st_size for a in archivos)/1024:.1f} KB")

## 4. Consultas interactivas con Spark SQL

In [ ]:
import sys; sys.path.insert(0, "bigdata/src")
from comun import cargar_config, crear_spark
from transformacion import registrar_vistas_oro
from consultas import CONSULTAS_EXPERIMENTO

cfg = cargar_config()
spark = crear_spark(cfg)
registrar_vistas_oro(spark, cfg)
for q in ["Q1", "Q2", "Q4"]:
    print(q); display(spark.sql(CONSULTAS_EXPERIMENTO[q].format(hechos="fact_ventas")).toPandas())
print("Spark UI:", spark.sparkContext.uiWebUrl)

In [ ]:
# Plan de ejecución de Q1: muestra el filtro de particiones y el broadcast de las dimensiones
spark.sql(CONSULTAS_EXPERIMENTO["Q1"].format(hechos="fact_ventas")).explain(mode="formatted")
spark.stop()

## 5. Experimento de tiempos (x1, x10, x50)

In [ ]:
!python bigdata/run_pipeline.py --etapas experimento

## 6. Comparación con SQL Server

Suba el archivo `sqlserver_resultados.csv` exportado de `dw.exp_resultados` (script `bi/sql/06_experimento_sqlserver.sql`).
Si no lo sube, el análisis muestra solo los tiempos de Spark.

In [ ]:
try:
    from google.colab import files
    subido = files.upload()
    for nombre, contenido in subido.items():
        open("experimento/resultados/sqlserver_resultados.csv", "wb").write(contenido)
except ImportError:
    print("Fuera de Colab: copie el CSV a experimento/resultados/ manualmente.")
!python experimento/analizar_resultados.py
from IPython.display import Image
Image("experimento/resultados/tiempos_por_consulta.png")